# 🎓 Hands-On Lab: Building a Production RAG System with Python
### *Retrieval-Augmented Generation (RAG) on Student Directory & Custom Documents in 60 Minutes*

---

## 🎯 What You Will Build Today
In this practical session, you will build an end-to-end **RAG (Retrieval-Augmented Generation)** pipeline. You will ingest structured student profile records from **`sample_data/student_info_clean.txt`**, chunk the profiles using custom card delimiters, convert them into dense vector embeddings, store them inside a Vector Database (**ChromaDB**), and query an LLM (**Google Gemini / Groq / OpenAI**) to retrieve student roll numbers, GitHub README links, sections, and campuses with exact precision.

```
+-------------------------------+       +--------------------------+       +----------------------+
| student_info_clean.txt Cards  | ----> | Card-Delimiter Splitter  | ----> | Embedding Model      |
+-------------------------------+       +--------------------------+       +-----------+----------+
                                                                                       |
                                                                                       v
+-------------------------------+       +--------------------------+       +----------------------+
| User Query (e.g. Roll No?)    | ----> | Top-K Vector Retrieval   | <---> | ChromaDB Vector DB   |
+-------------------------------+       +-------------+------------+       +----------------------+
                                                      | (Student Context)
                                                      v
                                         +-------------------------+
                                         | Prompt + Context + LLM  | ----> Grounded Student Record
                                         +-------------------------+
```


---
## 📦 Step 1: Install & Import Dependencies
Let's install the necessary packages using Jupyter's `%pip` magic (which ensures installation into the active Python kernel).

In [ ]:
# %pip ensures packages are installed into the exact active Jupyter kernel
%pip install -q langchain langchain-community langchain-core langchain-text-splitters chromadb pypdf sentence-transformers langchain-huggingface python-dotenv ipywidgets langchain-google-genai langchain-openai langchain-groq

In [54]:
import os
import getpass
import warnings

# Suppress migration warnings for clean classroom output
warnings.filterwarnings('ignore')

# Load .env file (with fallback if python-dotenv isn't loaded)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    if os.path.exists(".env"):
        with open(".env", "r") as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    k, v = line.split("=", 1)
                    os.environ[k.strip()] = v.strip().strip("'").strip('"')

print("✅ Environment initialized. Choose your LLM provider below.")

✅ Environment initialized. Choose your LLM provider below.


### 🔑 Configure Your LLM API Key
You can use **Google Gemini** (Free tier available at [Google AI Studio](https://aistudio.google.com/)), **Groq** (Free high-speed tier at [Groq Console](https://console.groq.com/keys)), or **OpenAI**.

In [55]:
# Prompt for API Key if not already set in environment
if not os.environ.get("GOOGLE_API_KEY") and not os.environ.get("OPENAI_API_KEY") and not os.environ.get("GROQ_API_KEY"):
    provider_choice = input("Enter provider (gemini / groq / openai) [default: gemini]: ").strip().lower() or "gemini"
    if provider_choice == "gemini":
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google Gemini API Key: ")
    elif provider_choice == "groq":
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")
    else:
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")

print("🔑 API Key successfully registered!")

🔑 API Key successfully registered!


---
## 📄 Step 2: Document Ingestion (Load `student_info_clean.txt`)
We load our structured student directory dataset using `TextLoader`.

In [56]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader

# Path to clean structured student directory
data_path = "sample_data/student_info_clean.txt"

if data_path.endswith(".pdf"):
    loader = PyPDFLoader(data_path)
else:
    loader = TextLoader(data_path, encoding="utf-8")

raw_docs = loader.load()

print(f"📄 Successfully loaded: {data_path}")
print(f"📑 Document Total Character Length: {len(raw_docs[0].page_content)}")
print(f"\n--- Preview of First Student Profile Card ---\n{raw_docs[0].page_content[:350]}...")

📄 Successfully loaded: sample_data/student_info_clean.txt
📑 Document Total Character Length: 29799

--- Preview of First Student Profile Card ---
STUDENT PROFILE
Name: Shreyansh Singh
Roll Number: 2501030152
Section: SOT25B2
Campus: Noida sec 126
Readme URL: https://github.com/shreyansh-singh-097/shreyansh-singh-097/blob/main/README.md
GitHub User: shreyansh-singh-097
Bio: None

============...


---
## ✂️ Step 3: Text Chunking by Profile Card Delimiters
### Why Card-Based Chunking?
In standard RAG, arbitrary character splits can cut a student's record in half (e.g. name in Chunk 1 and README link in Chunk 2).
By using `separators=["==================================================", "\n\n"]` and `chunk_size=350`, **each chunk is guaranteed to be exactly ONE self-contained student profile card** with zero boundary fragmentation!

In [57]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configure text splitter tailored to student_info_clean.txt card format
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,        # Maximum character count per student card
    chunk_overlap=0,       # 0 overlap because each card is complete and independent
    length_function=len,
    separators=["==================================================", "\n\n"]
)

chunks = text_splitter.split_documents(raw_docs)

print(f"🧩 Total Student Profile Chunks created: {len(chunks)}")
for idx, chunk in enumerate(chunks[:3], 1):
    print(f"\n[Chunk #{idx} | Length: {len(chunk.page_content)} chars]")
    print(chunk.page_content.strip())
    print("-"*50)

🧩 Total Student Profile Chunks created: 100

[Chunk #1 | Length: 336 chars]
STUDENT PROFILE
Name: Shreyansh Singh
Roll Number: 2501030152
Section: SOT25B2
Campus: Noida sec 126
Readme URL: https://github.com/shreyansh-singh-097/shreyansh-singh-097/blob/main/README.md
GitHub User: shreyansh-singh-097
Bio: None
--------------------------------------------------

[Chunk #2 | Length: 296 chars]
STUDENT PROFILE
Name: Anuj
Roll Number: 2501030012
Section: Batch-2
Campus: Noida
Readme URL: https://github.com/anujkochar21/anujkochar21/blob/main/README.md
GitHub User: anujkochar21
Bio: None
--------------------------------------------------

[Chunk #3 | Length: 277 chars]
STUDENT PROFILE
Name: Mayank
Roll Number: 2501030055
Section: B1
Campus: Noida
Readme URL: https://github.com/sahumayank108-bot/Mayank
GitHub User: sahumayank108-bot
Bio: None
--------------------------------------------------


---
## 🧠 Step 4: Vector Embeddings & Vector Database (ChromaDB)
### How Vector Retrieval Works:
1. **Embedding**: We pass student profile chunks into `sentence-transformers/all-MiniLM-L6-v2` locally to produce 384-dimensional dense vectors.
2. **ChromaDB**: An AI-native vector database that indexes vector embeddings for sub-millisecond similarity search.
3. **Cosine Similarity**: Finds student records whose vector angle is closest to the query.

In [58]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("⏳ Loading local embedding model (sentence-transformers/all-MiniLM-L6-v2)... (100% Free & Runs Locally)")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Initialize ChromaDB Vector Store with clean student chunks
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="students_clean_rag_collection"
)

print("✅ Vector database successfully created and indexed in ChromaDB!")

⏳ Loading local embedding model (sentence-transformers/all-MiniLM-L6-v2)... (100% Free & Runs Locally)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Vector database successfully created and indexed in ChromaDB!


### 🔍 Live Test: Raw Vector Similarity Search
Let's test similarity search directly against ChromaDB to see what raw student chunks are retrieved.

In [59]:
test_query = "What is the roll number of student Anuj?"

# Retrieve top 2 most relevant chunks with distance scores
results = vector_db.similarity_search_with_score(test_query, k=2)

print(f"🔎 Query: '{test_query}'\n")
for rank, (doc, score) in enumerate(results, start=1):
    print(f"🏅 Rank #{rank} (Distance Score: {score:.4f})")
    print(f"{doc.page_content.strip()}\n" + "-"*50)

🔎 Query: 'What is the roll number of student Anuj?'

🏅 Rank #1 (Distance Score: 0.7989)
STUDENT PROFILE
Name: Anuj
Roll Number: 2501030012
Section: Batch-2
Campus: Noida
Readme URL: https://github.com/anujkochar21/anujkochar21/blob/main/README.md
GitHub User: anujkochar21
Bio: None
--------------------------------------------------
🏅 Rank #2 (Distance Score: 0.8652)
STUDENT PROFILE
Name: Anshu Kumar
Roll Number: 2501010035
Section: B2
Campus: BLR
Readme URL: https://github.com/anshuks-byte/anshuks_byte
GitHub User: anshuks-byte
Bio: None
--------------------------------------------------


---
## 🤖 Step 5: Initializing the LLM (with High-Availability Auto-Retry)
We connect the LLM with automated retry and fallback support for maximum resilience.

In [60]:
# Initialize LLM with high resilience and automatic retries
if os.environ.get("GOOGLE_API_KEY"):
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm_primary = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0.1, max_retries=5)
    llm_fallback = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.1, max_retries=5)
    llm = llm_primary.with_fallbacks([llm_fallback])
    print("🤖 Using Google Gemini (High-Availability with Auto-Retry & Fallback)")
elif os.environ.get("GROQ_API_KEY"):
    from langchain_groq import ChatGroq
    llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.1, max_retries=3)
    print("🤖 Using Groq Llama 3.1 8B Instant")
elif os.environ.get("OPENAI_API_KEY"):
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1, max_retries=3)
    print("🤖 Using OpenAI GPT-4o-mini")
else:
    raise ValueError("Please provide an API key in Step 1!")

🤖 Using Google Gemini (High-Availability with Auto-Retry & Fallback)


---
## ⚡ Step 6: Constructing the Prompt & Building the RAG Chain
We formulate a strict student directory system prompt.

In [61]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Prompt Template
rag_prompt_template = """
You are an accurate Student Directory Assistant answering questions based on student records.
Strict Instructions:
1. Answer the question accurately using ONLY the context provided below.
2. If the info is not in the context, state 'I cannot find this information in the student records.'
3. Format student details clearly:
   - Name: ...
   - Roll Number: ...
   - Section: ...
   - Campus: ...
   - Readme URL: ...

Context:
{context}

Question: {question}

Answer:
"""

prompt = PromptTemplate(template=rag_prompt_template, input_variables=["context", "question"])

# 2. Retriever interface (Top-3 chunks)
retriever = vector_db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

# Format retrieved docs
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# 3. LangChain Expression Language (LCEL) Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("⛓️ Student RAG Chain successfully constructed!")

⛓️ Student RAG Chain successfully constructed!


---
## 💬 Step 7: Live RAG Queries on Student Information
Let's ask real-world questions to query student details!

In [62]:
def ask_rag(question: str):
    print(f"\n❓ QUESTION: {question}")
    source_docs = retriever.invoke(question)
    answer = rag_chain.invoke(question)
    
    print("\n💡 ANSWER:")
    print(answer)
    print("\n📚 RETRIEVED STUDENT CHUNKS:")
    for i, doc in enumerate(source_docs, 1):
        print(f"  [{i}] {doc.page_content[:100].strip()}...")
    print("="*60)

# Query 1: Find Roll Number
ask_rag("What is the roll number of student Anuj?")

# Query 2: Find GitHub Readme File
ask_rag("I want to know the readme file of student Mayank")

# Query 3: Find Campus & Section Details
ask_rag("I want to know the campus details and section of Nikhil Kumar")

# Query 4: Full Profile Query
ask_rag("What is the roll number, campus, section, and readme URL of Shreyansh Singh?")

# Query 5: Readme Link of another student
ask_rag("What is the readme file of Vansh Sahu?")

# Query 6: Specific Student Query
ask_rag("What is the roll number and campus of student Bhavesh Kumawat?")


❓ QUESTION: What is the roll number of student Anuj?

💡 ANSWER:
Name: Anuj
Roll Number: 2501030012
Section: Batch-2
Campus: Noida
Readme URL: https://github.com/anujkochar21/anujkochar21/blob/main/README.md

📚 RETRIEVED STUDENT CHUNKS:
  [1] ==================================================
STUDENT PROFILE
Name: Anuj
Roll Number: 250103001...
  [2] ==================================================
STUDENT PROFILE
Name: Anshu Kumar
Roll Number: 25...
  [3] ==================================================
STUDENT PROFILE
Name: prince kumar
Roll Number: 2...

❓ QUESTION: I want to know the readme file of student Mayank

💡 ANSWER:
- Name: Mayank
- Roll Number: 2501030055
- Section: B1
- Campus: Noida
- Readme URL: https://github.com/sahumayank108-bot/Mayank

📚 RETRIEVED STUDENT CHUNKS:
  [1] ==================================================
STUDENT PROFILE
Name: Mayank
Roll Number: 2501030...
  [2] ==================================================
STUDENT PROFILE
Name: SHIVAM CHAUDH

### 🛡️ Test Guardrails: Hallucination Prevention Check
What happens if we ask about a student who is **NOT** in the dataset?

In [63]:
# Student not in database
ask_rag("What is the roll number of Harry Potter at Hogwarts campus?")


❓ QUESTION: What is the roll number of Harry Potter at Hogwarts campus?

💡 ANSWER:
I cannot find this information in the student records.

📚 RETRIEVED STUDENT CHUNKS:
  [1] ==================================================
STUDENT PROFILE
Name: •Name : Shreyansh Singh
Rol...
  [2] ==================================================
STUDENT PROFILE
Name: prince kumar
Roll Number: 2...
  [3] ==================================================
STUDENT PROFILE
Name: Yatharth Singh
Roll Number:...


---
## 🌟 Step 8: Interactive Student Search Console
Type any student question interactively below!

In [64]:
# Interactive Search Console (100% Compatible with VS Code, Jupyter, and Colab)
def interactive_search():
    print("="*60)
    print("🔍 STUDENT DIRECTORY AI SEARCH CONSOLE")
    print("Type any student question below. Type 'exit' to quit.")
    print("="*60)
    
    default_q = "What is the roll number and campus of student Bhavesh Kumawat?"
    user_q = input(f"Enter question [Default: {default_q}]: ").strip()
    
    if not user_q:
        user_q = default_q
        
    if user_q.lower() in ['exit', 'quit', 'q']:
        print("👋 Exited search console.")
        return
        
    ask_rag(user_q)

# Run interactive search
interactive_search()

🔍 STUDENT DIRECTORY AI SEARCH CONSOLE
Type any student question below. Type 'exit' to quit.

❓ QUESTION: What are the details of Arin Rai

💡 ANSWER:
- Name: Arin Rai
- Roll Number: 2501030193
- Section: B1 (also listed as b1)
- Campus: Lucknow
- Readme URL: Not specified

📚 RETRIEVED STUDENT CHUNKS:
  [1] ==================================================
STUDENT PROFILE
Name: Arin Rai
Roll Number: 25010...
  [2] ==================================================
STUDENT PROFILE
Name: Arin Rai
Roll Number: 25010...
  [3] ==================================================

================================================...


---
## 🚀 Student Challenges (Mini-Hacks for Lab Practice)

Try these 3 exercises to deepen your understanding:

### 🎯 Challenge 1: Query by Roll Number
- Ask: *"Which student has roll number 2501040071 and what is their campus?"*
- Verify how vector embeddings handle reverse lookup (from ID to Name).

### 🎯 Challenge 2: Search by Campus
- Ask: *"List the students from Pune campus."*
- Increase $k$ (`search_kwargs={"k": 8}`) to see how many matching records can be retrieved simultaneously.

### 🎯 Challenge 3: Ingest Custom Submissions
- Add your own name, roll number, campus, and GitHub repository to `sample_data/student_info_clean.txt`.
- Re-index ChromaDB and ask the AI about yourself!